In [1]:
!pip install trafilatura spacy pandas httpx
!python -m spacy download en_core_web_trf

     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 1.3/457.4 MB 7.4 MB/s eta 0:01:02
     ---------------------------------------- 2.9/457.4 MB 7.3 MB/s eta 0:01:03
     ---------------------------------------- 4.2/457.4 MB 7.0 MB/s eta 0:01:05
     ---------------------------------------- 5.5/457.4 MB 6.4 MB/s eta 0:01:11
      --------------------------------------- 6.6/457.4 MB 6.4 MB/s eta 0:01:11
      --------------------------------------- 8.1/457.4 MB 6.5 MB/s eta 0:01:10
      --------------------------------------- 9.4/457.4 MB 6.5 MB/s eta 0:01:10
      -------------------------------------- 10.7/457.4 MB 6.5 MB/s eta 0:01:10
     - ------------------------------------- 11.8/457.4 MB 6.3 MB/s eta 0:01:11
     - ------------------------------------- 12.6/457.4 MB 6.1 MB/s eta 0:01:14
     - ------------------------------------- 13.4/457.4 MB 5.8 MB/s eta 0:01:17
     - ------------------------------------- 14

In [2]:
!pip install spacy-transformers

In [1]:
import httpx
import trafilatura
import spacy
import pandas as pd
import json

# Load the transformer model
# python -m spacy download en_core_web_trf
nlp = spacy.load("en_core_web_trf")

In [2]:
import httpx
import trafilatura
import json

# Target URLs related to your Knowledge Graph domain (Sci-Fi)
urls = [
    "https://en.wikipedia.org/wiki/Isaac_Asimov",
    "https://en.wikipedia.org/wiki/Frank_Herbert",
    "https://en.wikipedia.org/wiki/Arthur_C._Clarke",
    "https://en.wikipedia.org/wiki/Philip_K._Dick",
    "https://en.wikipedia.org/wiki/Ursula_K._Le_Guin",
    "https://en.wikipedia.org/wiki/Science_fiction",
    "https://en.wikipedia.org/wiki/Dune_(novel)",
    "https://fr.wikipedia.org/wiki/Victor_Hugo"
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

cleaned_data = []

# Empty the output file before appending new domain data
with open("crawler_output.jsonl", "w", encoding="utf-8") as f:
    pass

for url in urls:
    try:
        response = httpx.get(url, headers=headers, follow_redirects=True, timeout=10.0)
        content = trafilatura.extract(response.text)
        
        if content:
            word_count = len(content.split())
            print(f"URL: {url} | Words found: {word_count}") 
            
            if word_count > 500: 
                result = {"url": url, "text": content}
                cleaned_data.append(result)
                
                with open("crawler_output.jsonl", "a", encoding="utf-8") as f:
                    f.write(json.dumps(result) + "\n")
        else: 
            print(f"Trafilatura could not extract text from {url}")
            
    except Exception as e:
        print(f"Error for {url}: {e}")

print(f"\nTotal saved: {len(cleaned_data)} pages.")

URL: https://en.wikipedia.org/wiki/Isaac_Asimov | Words found: 22374
URL: https://en.wikipedia.org/wiki/Frank_Herbert | Words found: 6454
URL: https://en.wikipedia.org/wiki/Arthur_C._Clarke | Words found: 13287
URL: https://en.wikipedia.org/wiki/Philip_K._Dick | Words found: 13358
URL: https://en.wikipedia.org/wiki/Ursula_K._Le_Guin | Words found: 14608
URL: https://en.wikipedia.org/wiki/Science_fiction | Words found: 13925
URL: https://en.wikipedia.org/wiki/Dune_(novel) | Words found: 16577
URL: https://fr.wikipedia.org/wiki/Victor_Hugo | Words found: 26808

Total saved: 8 pages.


In [3]:
import pandas as pd
import spacy
import json

# Load the English transformer model
nlp = spacy.load("en_core_web_trf")

# Load the scraped data from the previous step
data = []
with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

extracted_entities = []

for entry in data:
    # Truncate text to avoid exceeding the transformer maximum sequence length
    raw_text = entry.get("text", "")
    safe_text = raw_text[:5000] 
    
    doc = nlp(safe_text)
    url = entry.get("url", "Unknown")
    
    for ent in doc.ents:
        # Keep relevant entity types including WORK_OF_ART for books
        if ent.label_ in ["PERSON", "ORG", "GPE", "WORK_OF_ART", "DATE"]:
            extracted_entities.append({
                "entity": ent.text.strip(),
                "type": ent.label_,
                "source_url": url
            })

# Remove duplicates to maintain a clean entity list
df_entities = pd.DataFrame(extracted_entities).drop_duplicates()

# Save the extracted entities to a new CSV file
output_file = "extracted_knowledge_scifi.csv"
df_entities.to_csv(output_file, index=False, encoding="utf-8")

print(f"Total unique entities extracted: {len(df_entities)}")
print("\nFirst 10 entities found:")
print(df_entities.head(10))

Total unique entities extracted: 558

First 10 entities found:
                  entity    type                                  source_url
0           Isaac Asimov  PERSON  https://en.wikipedia.org/wiki/Isaac_Asimov
2   c. January 2, 1920[a    DATE  https://en.wikipedia.org/wiki/Isaac_Asimov
3             Petrovichi  PERSON  https://en.wikipedia.org/wiki/Isaac_Asimov
4          Soviet Russia     GPE  https://en.wikipedia.org/wiki/Isaac_Asimov
5          April 6, 1992    DATE  https://en.wikipedia.org/wiki/Isaac_Asimov
6          New York City     GPE  https://en.wikipedia.org/wiki/Isaac_Asimov
7                   U.S.     GPE  https://en.wikipedia.org/wiki/Isaac_Asimov
8                     72    DATE  https://en.wikipedia.org/wiki/Isaac_Asimov
9          United States     GPE  https://en.wikipedia.org/wiki/Isaac_Asimov
10   Columbia University     ORG  https://en.wikipedia.org/wiki/Isaac_Asimov


In [4]:
extracted_entities = []

for entry in data:
    doc = nlp(entry["text"])
    url = entry["url"]
    
    # Extract specific entities as requested in the lab
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "DATE"]:
            extracted_entities.append({
                "entity": ent.text.strip(),
                "type": ent.label_,
                "source_url": url
            })

# Save to the required CSV format
df_entities = pd.DataFrame(extracted_entities).drop_duplicates()
df_entities.to_csv("extracted_knowledge.csv", index=False, encoding="utf-8")

In [5]:
triples = []

for entry in data:
    # Processing text in chunks if it is too long for the transformer model
    doc = nlp(entry["text"][:10000]) 
    
    for sent in doc.sents:
        entities_in_sent = [ent for ent in sent.ents if ent.label_ in ["PERSON", "ORG", "GPE"]]
        
        # If we have at least 2 entities, let's look for a verb connecting them
        if len(entities_in_sent) >= 2:
            for token in sent:
                # Check for a verb that has a subject and an object
                if token.pos_ == "VERB":
                    subj = [w for w in token.lefts if w.dep_ in ["nsubj", "nsubjpass"]]
                    obj = [w for w in token.rights if w.dep_ in ["dobj", "pobj"]]
                    
                    if subj and obj:
                        triples.append({
                            "subject": subj[0].text,
                            "relation": token.lemma_,
                            "object": obj[0].text
                        })

# Optional: display the first few triples found
print(pd.DataFrame(triples))

     subject   relation      object
0     Asimov      write   mysteries
1     Asimov      write      series
2    surname        end         -ov
3     father      spell          it
4       This    inspire         one
..       ...        ...         ...
130    titre    suggère         âge
131     Hugo     publie       roman
132      qui    entouré   rédaction
133  Foucher  rapproche      couple
134    Adèle      avoue  infidélité

[135 rows x 3 columns]
